### Adaptive algorithms

Some experimental algorithms to compute the derivative of a function iteratively

In [1]:
import numpy as np

In [2]:
def f1(x): # Target function
    return x ** 2 / (x ** 3 + 1.0)

In [3]:
def dev(f, x0, h, scheme): # Finite difference derivative
    match scheme:
        case 'forward':
            return (f(x0 + h) - f(x0)) / h
        case 'backward':
            return (f(x0) - f(x0 - h)) / h
        case 'central':
            return (f(x0 + h) - f(x0 - h)) / (2.0 * h)
        case _:
            return 0.0

In [4]:
def dev_adaptive(f, x0, h0 = 0.1, eps = 1e-06, scheme = 'forward'): # Adaptive finite difference

    h = 2.0 * h0
    j = 0
    err2 = eps + 1.0

    while err2 > eps: # Absolute error criterion

        h /= 2.0
        j += 1

        derivative1 = dev(f, x0, h, scheme)
        derivative2 = dev(f, x0, h / 2.0, scheme)
        diff21 = derivative2 - derivative1
        err2 = np.abs(diff21) if (scheme == 'forward' or scheme == 'backward') else np.abs(diff21) / 3.0 # Richardson error formula applied for derivative2

    rel_err2 = err2 / np.abs(derivative2) if derivative2 != 0.0 else np.nan # Relative error
    
    # Quality factor of the error. Better when the difference is close to zero
    derivative4 = dev(f, x0, h / 4.0, scheme)
    diff42 = derivative4 - derivative2
    ratio = diff21 / diff42 if diff42 != 0.0 else np.inf
    diffQ = np.abs(2.0 - ratio) if (scheme == 'forward' or scheme == 'backward') else np.abs(4.0 - ratio)

    print(f"Loop ended at j = {j}. h = {h}\n")
    return derivative2, err2, rel_err2, diffQ # It returns the derivative, the absolute error and relative error committed, the delta quality factor

In [5]:
x0 = 1.0 # Point target

In [6]:
dev_adaptive( # Forward scheme
    f = f1,
    x0 = x0,
    scheme = 'forward'
)

Loop ended at j = 16. h = 3.0517578125e-06



(0.24999904635478742,
 np.float64(9.536961442790926e-07),
 np.float64(3.8147991289760756e-06),
 np.float64(0.0005341880341882543))

In [7]:
dev_adaptive( # Backward scheme
    f = f1,
    x0 = x0,
    scheme = 'backward'
)

Loop ended at j = 16. h = 3.0517578125e-06



(0.2500009536743164,
 np.float64(9.53677954385057e-07),
 np.float64(3.814697265625e-06),
 np.float64(0.0001907523271782452))

In [8]:
dev_adaptive( # Central scheme
    f = f1,
    x0 = x0,
    scheme = 'central'
)

Loop ended at j = 6. h = 0.003125



(0.2500007629354428,
 np.float64(7.629194313333679e-07),
 np.float64(3.0516684124295055e-06),
 np.float64(7.855757793917562e-05))

In [9]:
dev_adaptive( # Central scheme with smaller epsilon
    f = f1,
    x0 = x0,
    eps = 1e-12,
    scheme = 'central'
)

Loop ended at j = 20. h = 1.9073486328125e-07



(0.24999999994179234, np.float64(0.0), np.float64(0.0), np.float64(4.0))

The quality factor is 0, that means the Richardson error estimation has failed. $\epsilon$ is too small. Let's implement a different adaptive version without $\epsilon$

In [10]:
def dev_adaptive_Q(f, x0, h0 = 0.1, scheme = 'forward', max_iter = 100):

    p = 1 if scheme in ('forward', 'backward') else 2

    # Initial triple of estimates on steps (h, h/2, h/4)
    h = h0
    d1 = dev(f, x0, h, scheme) # step h
    d2 = dev(f, x0, h / 2.0, scheme) # step h/2
    d4 = dev(f, x0, h / 4.0, scheme) # step h/4

    diff21 = d2 - d1
    diff42 = d4 - d2

    if diff42 == 0.0: # Q undefined -> stop the entire function
        print("h0 is too small. Initial degeneracy d4 == d2")
        return None

    ratio = (d2 - d1) / (d4 - d2)
    diffq_best = np.abs(2.0 ** p - ratio)

    h_best = h
    deriv_best = d1
    diff21_best = diff21

    status = 'maximum number of iterations reached'

    for j in range(1, max_iter + 1): # Halving loop reuses the two evaluations that the new window

        d1 = d2 # Old middle becomes new coarse
        d2 = d4 # Old fine becomes new middle
        
        h /= 2.0
        d4 = dev(f, x0, h / 4.0, scheme) # Only one new evaluation

        diff21 = d2 - d1
        diff42 = d4 - d2

        if diff42 == 0.0: # Q undefined -> stop
            status = 'degenerate d4 == d2. Loop stopped'
            break

        ratio = (d2 - d1) / (d4 - d2)
        diffq_new = np.abs(2.0 ** p - ratio)

        if diffq_new < diffq_best:
            diffq_best = diffq_new
            h_best = h
            deriv_best = d1
            diff21_best = diff21
        else:
            status = 'converged'
            break

    # Error estimates as Richardson extrapolation on d1
    err_abs = abs(diff21_best) * (2.0 ** p) / (2.0 ** p - 1.0)
    rel_err = err_abs / abs(deriv_best) if deriv_best != 0.0 else np.nan

    return {
        'iterations':     j,
        'h':              h_best,
        'derivative':     deriv_best,
        'absolute error': err_abs,
        'relative error': rel_err,
        'diffQ':          diffq_best,
        'status':         status,
    }

In [11]:
dev_adaptive_Q(
    f = f1,
    x0 = x0,
    scheme = 'forward'
)

{'iterations': 13,
 'h': 2.44140625e-05,
 'derivative': 0.2499847413992029,
 'absolute error': 1.5258528947015293e-05,
 'relative error': 6.10378411962705e-05,
 'diffQ': np.float64(3.5763332570937223e-06),
 'status': 'converged'}

In [12]:
dev_adaptive_Q(
    f = f1,
    x0 = x0,
    scheme = 'backward'
)

{'iterations': 14,
 'h': 1.220703125e-05,
 'derivative': 0.250007629438187,
 'absolute error': 7.629441824974492e-06,
 'relative error': 3.0516835994642433e-05,
 'diffQ': np.float64(4.7683306552137594e-06),
 'status': 'converged'}

In [13]:
dev_adaptive_Q(
    f = f1,
    x0 = x0,
    h0 = 1e-03, # Smaller h0
    scheme = 'central'
)

{'iterations': 1,
 'h': 0.001,
 'derivative': 0.2500003124992767,
 'absolute error': 3.124989896482096e-07,
 'relative error': 1.2499943961034599e-06,
 'diffQ': np.float64(8.526522221163901e-06),
 'status': 'converged'}

In [14]:
dev_adaptive_Q(
    f = f1,
    x0 = x0,
    h0 = 1e-06, # Smaller h0
    scheme = 'forward'
)

{'iterations': 1,
 'h': 1e-06,
 'derivative': 0.24999937497938163,
 'absolute error': 6.252776074688882e-07,
 'relative error': 2.5011166828735356e-06,
 'diffQ': np.float64(0.0),
 'status': 'converged'}

Let's test it on a fast oscillating function

In [15]:
def f2(x): # Hard function
    return np.sqrt(np.abs(np.sin(x ** 3)))

In [16]:
x0 = 2.0 * np.pi # New target point
h0 = 1e-03 # Initial step size

In [17]:
dev_adaptive_Q(
    f = f2,
    x0 = x0,
    h0 = h0,
    scheme = 'forward'
)

{'iterations': 12,
 'h': 4.8828125e-07,
 'derivative': np.float64(-159.5950844695153),
 'absolute error': np.float64(0.017556091506776283),
 'relative error': np.float64(0.00011000396136968567),
 'diffQ': np.float64(7.809920572965368e-05),
 'status': 'converged'}

In [18]:
dev_adaptive_Q(
    f = f2,
    x0 = x0,
    h0 = h0,
    scheme = 'backward'
)

{'iterations': 12,
 'h': 4.8828125e-07,
 'derivative': np.float64(-159.5599831008485),
 'absolute error': np.float64(0.01754535719555861),
 'relative error': np.float64(0.0001099608865242184),
 'diffQ': np.float64(2.412972680221337e-05),
 'status': 'converged'}

In [19]:
dev_adaptive_Q(
    f = f2,
    x0 = x0,
    h0 = h0,
    scheme = 'central'
)

{'iterations': 7,
 'h': 1.5625e-05,
 'derivative': np.float64(-159.58124800824967),
 'absolute error': np.float64(0.0037179636862371503),
 'relative error': np.float64(2.3298249215628063e-05),
 'diffQ': np.float64(0.0003060375299339668),
 'status': 'converged'}

Let's try to compute the derivative using brute force

In [20]:
def derivative_brute_force(f, x0, min = 2.0, max = 9.0, scheme = 'forward'): # Finds the minimum of diffQ. Then, computes the derivative
    
    # Scheme order p
    p = 1 if scheme in ('forward', 'backward') else 2

    # Array of h values
    hs = 10.0 ** (- np.arange(min, max + 1, 1e-04))

    # Array of diffQs
    diffQs = []

    for h in hs: # Computing diffQs for each h value
    
        derivative1 = dev(f, x0, h, scheme)
        derivative2 = dev(f, x0, h / 2.0, scheme)
        derivative4 = dev(f, x0, h / 4.0, scheme)

        diff21 = derivative2 - derivative1
        diff42 = derivative4 - derivative2

        if diff42 == 0.0:
            diffQs.append(np.nan)
        else:
            ratio = diff21 / diff42
            diffQs.append(np.abs(2.0 ** p - ratio))

    diffQs = np.array(diffQs)

    # Find best values
    min_diffQ = np.nanmin(diffQs) # Minimum diffQ found
    min_h = np.max(hs[diffQs == min_diffQ]) # best value of h found. Take the maximum to reduce roundoff error

    # Computing the derivative
    d1 = dev(f, x0, min_h, scheme)
    d2 = dev(f, x0, min_h / 2.0, scheme)
    err = np.abs(d2 - d1) * (2.0 ** p) / (2.0 ** p - 1.0) # Richardson error formula
    rel_err = err / np.abs(d1)

    info = {
        'derivative': d1,
        'absolute error': err,
        'relative error': rel_err,
        'h': min_h,
        'diff Q': min_diffQ
    }
    
    return info

Test on easy function

In [21]:
x0 = 1.0

In [22]:
derivative_brute_force(f1, x0, scheme = 'forward')

{'derivative': np.float64(0.24998551978843783),
 'absolute error': np.float64(1.4480131216187964e-05),
 'relative error': np.float64(5.792387986489163e-05),
 'h': np.float64(2.3168611115294866e-05),
 'diff Q': np.float64(0.0)}

In [23]:
derivative_brute_force(f1, x0, scheme = 'backward')

{'derivative': np.float64(0.2500097136668353),
 'absolute error': np.float64(9.713684402168354e-06),
 'relative error': np.float64(3.885322797942515e-05),
 'h': np.float64(1.5541752907503767e-05),
 'diff Q': np.float64(0.0)}

In [24]:
derivative_brute_force(f1, x0, scheme = 'central')

{'derivative': np.float64(0.2500004448859038),
 'absolute error': np.float64(4.448856030236925e-07),
 'relative error': np.float64(1.7795392453270682e-06),
 'h': np.float64(0.00119316361149832),
 'diff Q': np.float64(0.0)}

Test on hard function (it could fail badly)

In [25]:
x0 = 2.0 * np.pi # New target point for f2

In [26]:
derivative_brute_force(f2, x0, scheme = 'forward')

{'derivative': np.float64(-159.57895810332732),
 'absolute error': np.float64(0.0014279953442155602),
 'relative error': np.float64(8.948519035266126e-06),
 'h': np.float64(3.9755754491662004e-08),
 'diff Q': np.float64(0.0)}

In [27]:
derivative_brute_force(f2, x0, scheme = 'backward')

{'derivative': np.float64(-159.57628040101184),
 'absolute error': np.float64(0.0012491274694639287),
 'relative error': np.float64(7.827776573842288e-06),
 'h': np.float64(3.477763138455204e-08),
 'diff Q': np.float64(0.0)}

In [28]:
derivative_brute_force(f2, x0, scheme = 'central')

{'derivative': np.float64(-159.57760151217522),
 'absolute error': np.float64(7.135451232898049e-05),
 'relative error': np.float64(4.4714616370228114e-07),
 'h': np.float64(2.164711378664457e-06),
 'diff Q': np.float64(0.0)}